## Agent Skills with Claude Managed Agents

### Installing Utilities and Libraries

In [1]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Creating the Anthropic Client

In [3]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Upload the Marketing Skill

In [4]:
from anthropic.lib import files_from_dir

skill = client.beta.skills.create(
    files = files_from_dir("./skills/marketing-review")
)

print("Skill ID:", skill.id)
print("Latest Version:", skill.latest_version)

Skill ID: skill_01AWfEpWyrL5wJH7vFxGbdgE
Latest Version: 1790006081711349


### Creating the Agent

In [5]:
agent = client.beta.agents.create(
    name="Marketing-Agent",
    model=claude_model_name,
    system="You are a helpful AI Marketing Assistant.",
    skills = [
        {
            "type": "anthropic",
            "skill_id": "docx" 
        },
        {
            "type": "custom",
            "skill_id": skill.id,
            "version": skill.latest_version
        }
    ],
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

Agent ID: agent_01YNu8vtEJ4auuqfE26Espvt, version: 1


### Create an Environment

In [6]:
environment = client.beta.environments.create(
    name="quickstart-env-v2",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

Environment ID: env_01GiVdEzCMfk54s1Htv6MzYm


### Start a Session

In [7]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Quickstart session v2",
)

print(f"Session ID: {session.id}")

Session ID: sesn_01Mt83hMtD4W5DVhKxWuoTdK


### Execute the Agent

In [8]:
with client.beta.sessions.events.stream(session.id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session.id,
                betas = ["skills-2025-10-02"],
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                                "type": "text",
                                "text": """Create a professional LinkedIn product launch announcement for our new AI-powered fitness smartwatch, FitSense AI.

Use the company's branding guidelines, marketing checklist, and LinkedIn launch template to produce polished marketing content.

Generate the final deliverable as a professionally formatted Microsoft Word (.docx) document and save it as an output file.""",
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break


[Using tool: read]

[Using tool: bash]

[Using tool: read]

[Using tool: read]

[Using tool: read]
Now let me read the docx skill for creating the Word document.
[Using tool: read]

[Using tool: bash]
Now I'll create the Word document generation script following the docx-js skill guidance.
[Using tool: write]

[Using tool: bash]
Now let's verify the output by converting to PDF and viewing it.
[Using tool: bash]

[Using tool: read]
This looks polished and clean. It follows the template structure, brand guidelines (professional tone, "AI-powered" instead of "Revolutionary," single CTA "Learn more today"), and checklist (short paragraphs, scannable, no exaggerated claims). Let me copy it to the outputs directory.
[Using tool: bash]
I've created the LinkedIn product launch announcement for **FitSense AI** and saved it as a formatted Word document: **`FitSense_AI_LinkedIn_Launch_Announcement.docx`**

### How it was built
I followed the marketing-review skill's full workflow:
1. **Brand gui

In [9]:
files = client.beta.files.list(
    scope_id=session.id,
    betas=["managed-agents-2026-04-01"],
)

for file in files.data:

    if file.filename.endswith(".docx"):

        print(f"Downloading {file.filename}...")

        metadata = client.beta.files.retrieve_metadata(file.id)

        content = client.beta.files.download(file.id)

        content.write_to_file(metadata.filename)

        print("Done!")

Done!
